In [1]:
from __future__ import annotations

import re
from pathlib import Path
from typing import List, Dict, Any

import fitz  # PyMuPDF
from dotenv import load_dotenv
from langchain_core.documents import Document
# from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import pdfplumber

BASE_DIR = Path('C:/skn24/allianz-rag')

In [17]:
DATA_DIR = BASE_DIR / "data" / "raw"
DB_DIR = BASE_DIR / "vectordb"
COLLECTION_NAME = "allianz_care"

ENV_PATH = BASE_DIR / ".env"
load_dotenv(dotenv_path=ENV_PATH)
# 1. 파일 목록
FILES: List[Dict[str, Any]] = [
    # 글로벌 공통
    # {
    #     "path": DATA_DIR / "DOC-Care-IBG-EN-1125_개인고객용혜택가이드.pdf",
    #     "doc_type": "benefit_guide",
    #     "doc_year": 2025,
    #     "region": "global",
    #     "product_family": "care_global",
    # },
    {
        "path": DATA_DIR / "care-tob-en_보장금액.pdf",
        "doc_type": "tob",
        "doc_year": 2025,
        "region": "global",
        "product_family": "care_global",
    },
    # {
    #     "path": DATA_DIR / "FRM-PreAuth-EN-0825_사전승인신청서.pdf",
    #     "doc_type": "preauth_form",
    #     "doc_year": 2025,
    #     "region": "global",
    #     "product_family": "care_global",
    # },
    # {
    #     "path": DATA_DIR / "FRM-PCF-EN-1125_사후보험청구서.pdf",
    #     "doc_type": "claim_form",
    #     "doc_year": 2025,
    #     "region": "global",
    #     "product_family": "care_global",
    # },

    # # 지역별 코퍼스
    # {
    #     "path": DATA_DIR / "DOC-Singapore-IBG-EN-0126_싱가포르.pdf",
    #     "doc_type": "benefit_guide",
    #     "doc_year": 2026,
    #     "region": "singapore",
    #     "product_family": "regional",
    # },
    # {
    #     "path": DATA_DIR / "DOC-IBG-Dubai-Northern-Emirates-EN-0126_두바이.pdf",
    #     "doc_type": "benefit_guide",
    #     "doc_year": 2026,
    #     "region": "dubai_northern_emirates",
    #     "product_family": "regional",
    # },
    # {
    #     "path": DATA_DIR / "DOC-LEBANON-IBG-EN-0725_레바논.pdf",
    #     "doc_type": "benefit_guide",
    #     "doc_year": 2025,
    #     "region": "lebanon",
    #     "product_family": "regional",
    # },
    # {
    #     "path": DATA_DIR / "DOC-IBG-Indonesia-en-UK-1123_인도네시아.pdf",
    #     "doc_type": "benefit_guide",
    #     "doc_year": 2024,
    #     "region": "indonesia",
    #     "product_family": "regional",
    # },
    # {
    #     "path": DATA_DIR / "DOC-IBG-Vietnam-en-UK-0823_베트남.pdf",
    #     "doc_type": "benefit_guide",
    #     "doc_year": 2023,
    #     "region": "vietnam",
    #     "product_family": "regional",
    # },
    # {
    #     "path": DATA_DIR / "DOC-IBG-HongKong-en-UK-2024_홍콩.pdf",
    #     "doc_type": "benefit_guide",
    #     "doc_year": 2023,
    #     "region": "hong_kong",
    #     "product_family": "regional",
    # },
    # {
    #     "path": DATA_DIR / "DOC-IBG-AZJD-en-UK-0824_중국.pdf",
    #     "doc_type": "benefit_guide",
    #     "doc_year": 2024,
    #     "region": "china",
    #     "product_family": "regional",
    # },
    # {
    #     "path": DATA_DIR / "DOC-SUISSE-IBG-KPT-EN-0624_스위스.pdf",
    #     "doc_type": "benefit_guide",
    #     "doc_year": 2022,
    #     "region": "switzerland",
    #     "product_family": "regional",
    # },
    # {
    #     "path": DATA_DIR / "DOC-IBG-CARE-UK-EN-1125_영국.pdf",
    #     "doc_type": "benefit_guide",
    #     "doc_year": 2025,
    #     "region": "uk",
    #     "product_family": "regional",
    # },
    # {
    #     "path": DATA_DIR / "DOC-IBG-FP-en-UK-1223_프랑스.pdf",
    #     "doc_type": "benefit_guide",
    #     "doc_year": 2024,
    #     "region": "france_benelux_monaco",
    #     "product_family": "regional",
    # },
    # {
    #     "path": DATA_DIR / "DOC-Global-IBG-EN-0524_남미.pdf",
    #     "doc_type": "benefit_guide",
    #     "doc_year": 2024,
    #     "region": "latin_america",
    #     "product_family": "regional",
    # },
]


In [18]:
def clean_text(text: str) -> str:
    text = text.replace("\xa0", " ")
    text = text.replace("\u200b", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def read_pdf_pages(pdf_path: Path) -> List[tuple[int, str]]: 
    if not pdf_path.exists():
        print(f"[WARN] 파일이 없습니다: {pdf_path}")
        return []

    pages: List[tuple[int, str]] = []
    doc = fitz.open(pdf_path)

    try:
        for i, page in enumerate(doc):
            text = clean_text(page.get_text("text"))
            if text:
                pages.append((i + 1, text))
    finally:
        doc.close()

    return pages
# 4. 공통 메타데이터 생성
# 인덱싱된 각 문서 조각이 어떤 문서에서 왔는지,
# 어떤 유형의 문서인지, 어느 지역과 관련된 것인지 등의 정보를 담아줌.
def build_common_metadata(
    file_info: Dict[str, Any],
    source_name: str,
    page_num: int,
    chunk_idx: int | None = None,
    section: str | None = None,
) -> Dict[str, Any]:    # 리턴형식 : Dict[str, Any]
    metadata = {
        "source": source_name,
        "doc_type": file_info["doc_type"],
        "doc_year": file_info["doc_year"],
        "region": file_info["region"],
        "product_family": file_info["product_family"],
        "page": page_num,
        "insurer": "Allianz",
    }

    # chunk_idx가 있는 경우 : metadata에 chunk_idx 추가
    # chunk_idx는 같은 페이지 내에서 여러 개의 텍스트 조각이 나올 때, 각 조각을 구분하기 위한 인덱스입니다.
    if chunk_idx is not None:
        metadata["chunk_idx"] = chunk_idx

    # section이 있는 경우 : metadata에 section 추가
    # section은 문서 내에서 특정 섹션이나 제목을 나타내는 문자열입니다.
    # 예를 들어, "Coverage Details" 같은 섹션명이 될 수 있습니다.
    if section:
        metadata["section"] = section

    return metadata

# 5. 검색 보조 태그
REGION_ALIASES = {
    "global": ["global", "worldwide", "전세계", "글로벌", "공통"],
    "singapore": ["singapore", "싱가포르"],
    "dubai_northern_emirates": ["dubai", "northern emirates", "두바이", "북부에미리트", "uae", "아랍에미리트"],
    "lebanon": ["lebanon", "레바논"],
    "indonesia": ["indonesia", "인도네시아"],
    "vietnam": ["vietnam", "베트남"],
    "hong_kong": ["hong kong", "hk", "홍콩"],
    "china": ["china", "중국", "중화권"],
    "switzerland": ["switzerland", "suisse", "스위스"],
    "uk": ["uk", "united kingdom", "england", "britain", "영국"],
    "france_benelux_monaco": ["france", "benelux", "monaco", "프랑스", "모나코", "베네룩스"],
    "latin_america": ["latin america", "남미", "라틴아메리카"],
}

DOC_TYPE_ALIASES = {
    "benefit_guide": [
        "benefit guide", "coverage guide", "benefits", "혜택 가이드", "보장 안내", "보장", "혜택"
    ],
    "tob": [
        "table of benefits", "schedule of benefits", "benefit limits",
        "보장금액", "보장표", "한도표", "한도", "limit"
    ],
    "preauth_form": [
        "pre-authorisation form", "preauthorization form", "preauth form",
        "사전승인 신청서", "사전승인", "입원 전 승인", "직접청구 준비"
    ],
    "claim_form": [
        "claim form", "reimbursement form",
        "보험금 청구서", "청구서", "환급 청구", "사후 청구"
    ],
}

INSURANCE_SEARCH_TAGS = [
    "coverage", "covered", "benefit", "limit", "co-payment", "copay",
    "deductible", "waiting period", "exclusion", "outpatient", "inpatient",
    "maternity", "cancer", "chronic condition", "pre-existing condition",
    "pre-authorisation", "preauthorization", "planned hospitalisation",
    "direct billing", "claim", "reimbursement", "invoice", "receipt",
    "서류", "청구", "환급", "직접청구", "사전승인", "보장", "혜택",
    "한도", "면책", "제외사항", "외래", "입원", "출산", "기왕증"
]

# 검색 태그 빌드
def build_search_tags(file_info: Dict[str, Any]) -> str:
    region_aliases = REGION_ALIASES.get(file_info["region"], [])
    doc_type_aliases = DOC_TYPE_ALIASES.get(file_info["doc_type"], [])

    return "\n".join([
        "[search_tags]",
        f"region: {' | '.join(region_aliases)}",
        f"doc_type: {' | '.join(doc_type_aliases)}",
        f"product_family: {file_info['product_family']}",
        f"insurer: Allianz 알리안츠",
        "keywords: " + ", ".join(INSURANCE_SEARCH_TAGS),
    ])


def enrich_text_for_multilingual_search(text: str, file_info: Dict[str, Any]) -> str:
    tags = build_search_tags(file_info)
    return f"{text}\n\n{tags}"

In [19]:
def build_documents() -> List[Document]:
    all_docs: List[Document] = []

    for file_info in FILES:
        path: Path = file_info["path"]
        pages = read_pdf_pages(path)    # 파일에서 페이지 읽기

        if not pages:
            continue

        source_name = path.name
        doc_type = file_info["doc_type"]

        print(
            f"[INFO] 처리 중: {source_name} | type={doc_type} | "
            f"region={file_info['region']} | year={file_info['doc_year']}"
        )

        # if doc_type == "benefit_guide":
        #     docs = chunk_benefit_guide(pages, source_name, file_info)
        # elif doc_type == "tob":
        docs = chunk_tob_pdfplumber(path, source_name, file_info)
        # elif doc_type in ["preauth_form", "claim_form"]:
        #     docs = chunk_form(pages, source_name, file_info)
        # else:
        #     print(f"[WARN] 지원하지 않는 doc_type: {doc_type}")
        #     continue

        all_docs.extend(docs)

    return all_docs

In [20]:
# 7. TOB 청킹을 위한 헬퍼 함수
def normalize_cell_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\xa0", " ")
    text = text.replace("\u200b", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()

# TOB 문서에서 청킹할 때, 실제 benefit row로 보이는 줄과 그렇지 않은 줄을 구분하기 위한 헬퍼 함수
def is_noise_line(line: str) -> bool:
    line_l = line.lower().strip()

    noise_patterns = [
        r"^care base care enhanced care signature$",
        r"^core plans key to table of benefits",
        r"^√ covered in full",
        r"^x not available$",
        r"^maximum plan limit$",
        r"^out-patient plans$",
        r"^dental plans$",
        r"^area of cover$",
        r"^click here or press enter",
        r"^looking for something specific\?$",
    ]
    return any(re.search(p, line_l) for p in noise_patterns)

# TOB 문서에서 benefit row의 시작으로 보이는 줄인지 판단하는 함수입니다.
def is_section_header(line: str) -> bool:
    line_l = line.lower().strip()
    section_patterns = [
        r"^in-patient benefits$",
        r"^other benefits$",
        r"^additional core plan services$",
        r"^out-patient plan benefits$",
        r"^dental plan benefits$",
        r"^core plans\b",
        r"^out-patient plans\b",
        r"^dental plans\b",
    ]
    return any(re.search(p, line_l) for p in section_patterns)

# TOB 문서에서 benefit row의 시작으로 보이는 줄인지 판단하는 함수입니다.
def looks_like_row_start(line: str) -> bool:
    """
    benefit row 시작처럼 보이는 줄인지 판단
    """
    s = line.strip()
    s_l = s.lower()

    if not s:
        return False

    # 설명/값/조건 줄은 제외
    banned_fragments = [
        "pre-authorisation required",
        "waiting period applies",
        "in-patient and out-patient treatment",
        "in-patient treatment only",
        "in-patient and day-care treatment only",
        "out-patient treatment only",
        "day-care treatment",
        "per pregnancy",
        "per event",
        "max.",
        "private room",
        "up to",
        "√",
        "x",
        "£",
        "€",
        "us$",
        "chf",
    ]
    if any(x in s_l for x in banned_fragments):
        return False

    # 너무 긴 문장형 설명 제외
    if len(s) > 120:
        return False

    # benefit명처럼 보이는 패턴
    return bool(re.match(r"^[A-Z][A-Za-z0-9/\-\(\),&’' ]+$", s))

# TOB 문서의 페이지 텍스트에서 benefit row로 보이는 줄들을 찾아서, 각 row를 하나의 Document로 만들어 반환하는 함수입니다.
def build_tob_row_documents_from_page_text(
    text: str,
    page_num: int,
    source_name: str,
    file_info: Dict[str, Any],
) -> List[Document]:
    docs: List[Document] = []
    # 텍스트를 줄 단위로 나누고, 각 줄을 정리한 후, 빈 줄은 제거합니다.
    lines = [normalize_cell_text(line) for line in text.split("\n") if normalize_cell_text(line)]

    current_section = None
    current_row: List[str] = []
    chunk_idx = 0

    def flush_row():
        nonlocal chunk_idx, current_row
        if not current_row:
            return

        row_text = " | ".join(current_row).strip()
        if len(row_text) < 20:
            current_row = []
            return

        metadata = build_common_metadata(
            file_info=file_info,
            source_name=source_name,
            page_num=page_num,
            chunk_idx=chunk_idx,
            section=current_section,
        )

        structured_text = row_text
        if current_section:
            structured_text = f"Section: {current_section}\n{row_text}"

        docs.append(
            Document(
                page_content=enrich_text_for_multilingual_search(structured_text, file_info),
                metadata=metadata,
            )
        )
        chunk_idx += 1
        current_row = []

    for line in lines:
        if is_noise_line(line):
            continue

        if is_section_header(line):
            flush_row()
            current_section = line
            continue

        if looks_like_row_start(line):
            flush_row()
            current_row = [line]
            continue

        if current_row:
            current_row.append(line)
        else:
            # row 시작 전인데 설명성 텍스트가 있으면 section 밑 참고 텍스트로 버림
            continue

    flush_row()
    return docs


def chunk_tob_pdfplumber(
    pdf_path: Path,
    source_name: str,
    file_info: Dict[str, Any],
) -> List[Document]:
    docs: List[Document] = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_idx, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            text = normalize_cell_text(text)
            if not text:
                continue

            page_docs = build_tob_row_documents_from_page_text(
                text=text,
                page_num=page_idx,
                source_name=source_name,
                file_info=file_info,
            )
            docs.extend(page_docs)

    return docs

In [22]:
documents = build_documents()
documents

[INFO] 처리 중: care-tob-en_보장금액.pdf | type=tob | region=global | year=2025


[Document(metadata={'source': 'care-tob-en_보장금액.pdf', 'doc_type': 'tob', 'doc_year': 2025, 'region': 'global', 'product_family': 'care_global', 'page': 1, 'insurer': 'Allianz', 'chunk_idx': 0}, page_content='International healthcare plans | for you and your family\n\n[search_tags]\nregion: global | worldwide | 전세계 | 글로벌 | 공통\ndoc_type: table of benefits | schedule of benefits | benefit limits | 보장금액 | 보장표 | 한도표 | 한도 | limit\nproduct_family: care_global\ninsurer: Allianz 알리안츠\nkeywords: coverage, covered, benefit, limit, co-payment, copay, deductible, waiting period, exclusion, outpatient, inpatient, maternity, cancer, chronic condition, pre-existing condition, pre-authorisation, preauthorization, planned hospitalisation, direct billing, claim, reimbursement, invoice, receipt, 서류, 청구, 환급, 직접청구, 사전승인, 보장, 혜택, 한도, 면책, 제외사항, 외래, 입원, 출산, 기왕증'),
 Document(metadata={'source': 'care-tob-en_보장금액.pdf', 'doc_type': 'tob', 'doc_year': 2025, 'region': 'global', 'product_family': 'care_global', 'pag

In [23]:
print(len(documents))
print(type(documents[0]))
print(documents[0].page_content[:500])
print(documents[0].metadata)

94
<class 'langchain_core.documents.base.Document'>
International healthcare plans | for you and your family

[search_tags]
region: global | worldwide | 전세계 | 글로벌 | 공통
doc_type: table of benefits | schedule of benefits | benefit limits | 보장금액 | 보장표 | 한도표 | 한도 | limit
product_family: care_global
insurer: Allianz 알리안츠
keywords: coverage, covered, benefit, limit, co-payment, copay, deductible, waiting period, exclusion, outpatient, inpatient, maternity, cancer, chronic condition, pre-existing condition, pre-authorisation, preauthorization, planned h
{'source': 'care-tob-en_보장금액.pdf', 'doc_type': 'tob', 'doc_year': 2025, 'region': 'global', 'product_family': 'care_global', 'page': 1, 'insurer': 'Allianz', 'chunk_idx': 0}


In [26]:
for i, doc in enumerate(documents[30:40]):
    print(f"\n--- Chunk {i} ---")
    print(doc.page_content)
    print(doc.metadata)


--- Chunk 0 ---
Section: Other benefits
Rehabilitation treatment | In-patient, day-care and out-patient treatment; must commence within € 3,000 / € 7,500 / √ | US$ 4,050 / US$ 10,125 / | 14 days of discharge after the acute medical and/or surgical treatment max. 90 days | CHF 3,900, CHF 9,750, | ceases | max. 30 days max. 60 days

[search_tags]
region: global | worldwide | 전세계 | 글로벌 | 공통
doc_type: table of benefits | schedule of benefits | benefit limits | 보장금액 | 보장표 | 한도표 | 한도 | limit
product_family: care_global
insurer: Allianz 알리안츠
keywords: coverage, covered, benefit, limit, co-payment, copay, deductible, waiting period, exclusion, outpatient, inpatient, maternity, cancer, chronic condition, pre-existing condition, pre-authorisation, preauthorization, planned hospitalisation, direct billing, claim, reimbursement, invoice, receipt, 서류, 청구, 환급, 직접청구, 사전승인, 보장, 혜택, 한도, 면책, 제외사항, 외래, 입원, 출산, 기왕증
{'source': 'care-tob-en_보장금액.pdf', 'doc_type': 'tob', 'doc_year': 2025, 'region': 'global'

In [25]:
lengths = [len(doc.page_content) for doc in documents]
print("avg:", sum(lengths)/len(lengths))
print("min:", min(lengths))
print("max:", max(lengths))

avg: 801.2978723404256
min: 599
max: 1674


In [ ]:
queries = [
    "싱가포르 입원 사전 승인 필요?",
    "Is prior approval required for inpatient treatment in Singapore?",
    "新加坡住院需要事前批准吗？"
]

for q in queries:
    print(f"\n### Query: {q}")
    results = vectordb.similarity_search(q, k=2)
    for r in results:
        print(r.page_content[:200])